In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from src import config as C
from src.models import lightgbm_model, metrics
from src.policy import policy_params

engine = C.get_engine()
feat = pd.read_sql(text("SELECT * FROM features_weekly"), engine, parse_dates=["week_start"])
feat = feat.dropna(subset=["lag_52"]).reset_index(drop=True)

# Rolling origin: fit on weeks before 2016, validate on 2016. Test year 2017 is not touched.
pre_test = feat[feat["data_split"] == "train"]
fit = pre_test[pre_test["week_start"] < C.VAL_START]
val = pre_test[pre_test["week_start"] >= C.VAL_START]
print("fit:", fit.shape, " val:", val.shape)

val_fc = lightgbm_model(fit, val)
print({k: round(v, 4) for k, v in metrics(val_fc).items()})

fit: (52000, 13)  val: (26000, 13)
{'WMAPE': np.float64(0.0487), 'MAPE': np.float64(0.0544), 'RMSE': 25.5032, 'Bias': np.float64(-0.0149)}


In [2]:
dim_item = pd.read_sql(text("SELECT * FROM dim_item"), engine)
params = policy_params(val_fc, dim_item, C.SERVICE_LEVEL)

params.to_sql("policy_params", engine, if_exists="replace", index=False)
params.to_csv(C.OUT_DIR / "06_policy_params.csv", index=False)

print(params.shape)
print(params[["sigma_week", "bias_week", "annual_demand", "safety_stock", "eoq"]].describe().round(2))

(500, 12)
       sigma_week  bias_week  annual_demand  safety_stock      eoq
count      500.00     500.00         500.00        500.00   500.00
mean        23.44      -5.93       20633.04         41.61  1535.05
std          6.62       5.80        9503.88         14.85   655.24
min          9.88     -29.35        5041.00         13.51   553.00
25%         18.19      -7.74       12369.50         30.35  1102.46
50%         23.05      -3.87       19654.50         38.75  1399.61
75%         28.35      -2.24       27380.50         51.32  1782.87
max         45.52       1.93       44293.00         93.86  4824.10
